In [ ]:
container_name = "ecommerce"
account_name = "duongbambo"
credential = "bambo"

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bambo.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS bambo.gold")

print("Schemas created: bronze, silver, gold")

In [ ]:
tables = {
    "raw_orders":     "olist_orders_dataset.csv",
    "raw_customers":  "olist_customers_dataset.csv",
    "raw_order_items":"olist_order_items_dataset.csv",
    "raw_payments":   "olist_order_payments_dataset.csv",
    "raw_reviews":    "olist_order_reviews_dataset.csv",
    "raw_products":   "olist_products_dataset.csv",
    "raw_sellers":    "olist_sellers_dataset.csv",
    "raw_geolocation":"olist_geolocation_dataset.csv",
    "raw_categories": "product_category_name_translation.csv",
}

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

for table_name, file_name in tables.items():
    print(f"Processing {table_name}...")
    
    # Đọc CSV
    df = spark.read.csv(
        f"abfss://{container_name}@{account_name}.dfs.core.windows.net/bronze/csv/{file_name}",
        header=True,
        inferSchema=True
    )
    
    # Lưu Delta
    df.write.format("delta").mode("overwrite").save(
        f"abfss://ecommerce@duongbambo.dfs.core.windows.net/bronze/delta/{table_name}"
    )
    
    # Tạo table
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze.{table_name}
        USING DELTA
        LOCATION 'abfss://ecommerce@duongbambo.dfs.core.windows.net/bronze/delta/{table_name}'
    """)
    
    print(f"Done: bronze.{table_name}")

print("All tables loaded!")